# RetailPulse: Smart Metadata-Driven Retail Supply Chain Pipeline

## Enterprise Pipeline using Databricks, PySpark & Delta Lake

## Pipeline Architecture

    CSV Files
         │
         ▼
    Bronze Layer (Raw Data)
         │
         ▼
    Silver Layer (Cleaned Data)
         │
         ▼
    Gold Layer (Business KPIs)
         │
         ▼
    Business Analytics


In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

In [0]:
spark.sql("CREATE DATABASE IF NOT EXISTS retail_pulse_db")

spark.sql("USE retail_pulse_db")

DataFrame[]

In [0]:
SOURCE_PATH = "/Volumes/workspace/default/datasets/RetailPulse/"

In [0]:
orders = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv(SOURCE_PATH + "orders.csv")

products = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv(SOURCE_PATH + "products.csv")

stores = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv(SOURCE_PATH + "stores.csv")

suppliers = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv(SOURCE_PATH + "suppliers.csv")

inventory = spark.read \
    .option("header", True) \
    .option("inferSchema", True) \
    .csv(SOURCE_PATH + "inventory_.csv")

## Data Validation

Verify the schema, preview the records, and validate the number of rows before processing.

In [0]:
orders.printSchema()

display(orders)

print("Total Orders:", orders.count())

root
 |-- order_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- store_id: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- price: double (nullable = true)
 |-- order_date: date (nullable = true)



order_id,product_id,store_id,quantity,price,order_date
ORD-0001,PROD-018,STR-04,15,364.81,2024-01-16
ORD-0002,PROD-015,STR-03,13,238.6,2024-02-29
ORD-0003,PROD-022,STR-09,9,350.06,2024-03-17
ORD-0004,PROD-011,STR-08,10,42.18,2024-03-05
ORD-0005,PROD-014,STR-09,8,388.87,2024-01-21
ORD-0006,PROD-024,STR-08,8,433.01,2024-02-03
ORD-0007,PROD-025,STR-04,14,445.02,2024-03-22
ORD-0008,PROD-009,STR-09,8,407.15,2024-03-21
ORD-0009,PROD-008,STR-05,8,169.0,2024-01-10
ORD-0010,PROD-023,STR-05,4,448.48,2024-02-04


Total Orders: 600


## Bronze Layer

Store the raw datasets as Delta Lake tables without applying any transformations.  
The Bronze layer preserves the original source data for auditing and reprocessing.

In [0]:
orders.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("bronze_orders")

products.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("bronze_products")

stores.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("bronze_stores")

suppliers.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("bronze_suppliers")

inventory.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("bronze_inventory")

## Verify Bronze Tables

Confirm that all Bronze tables have been created successfully in the database.

In [0]:
spark.sql("SHOW TABLES").show(truncate=False)

+---------------+----------------+-----------+
|database       |tableName       |isTemporary|
+---------------+----------------+-----------+
|retail_pulse_db|bronze_inventory|false      |
|retail_pulse_db|bronze_orders   |false      |
|retail_pulse_db|bronze_products |false      |
|retail_pulse_db|bronze_stores   |false      |
|retail_pulse_db|bronze_suppliers|false      |
+---------------+----------------+-----------+



## Preview Bronze Orders Table

Read the Bronze Delta table and verify that the data has been stored correctly.

In [0]:
bronze_orders = spark.table("bronze_orders")

display(bronze_orders)

print("Total Bronze Orders:", bronze_orders.count())

order_id,product_id,store_id,quantity,price,order_date
ORD-0001,PROD-018,STR-04,15,364.81,2024-01-16
ORD-0002,PROD-015,STR-03,13,238.6,2024-02-29
ORD-0003,PROD-022,STR-09,9,350.06,2024-03-17
ORD-0004,PROD-011,STR-08,10,42.18,2024-03-05
ORD-0005,PROD-014,STR-09,8,388.87,2024-01-21
ORD-0006,PROD-024,STR-08,8,433.01,2024-02-03
ORD-0007,PROD-025,STR-04,14,445.02,2024-03-22
ORD-0008,PROD-009,STR-09,8,407.15,2024-03-21
ORD-0009,PROD-008,STR-05,8,169.0,2024-01-10
ORD-0010,PROD-023,STR-05,4,448.48,2024-02-04


Total Bronze Orders: 600


## Inspect Delta Table Metadata

View detailed information about the Bronze Delta table.

In [0]:
spark.sql("DESCRIBE DETAIL bronze_orders").show(truncate=False)

+------+------------------------------------+---------------------------------------+-----------+--------+-----------------------+-------------------+----------------+-----------------+--------+-----------+---------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----------------+----------------+-----------------------------------------+---------------------------------------------------------------+-------------+
|format|id                                  |name                                   |description|location|createdAt              |lastModified       |partitionColumns|clusteringColumns|numFiles|sizeInBytes|properties                                                                                                                                                                 |minReaderVersion|minWriterVersion|tableFeatures                            |statistics   

## View Delta Transaction History

Check the transaction history maintained by Delta Lake.

In [0]:
spark.sql("DESCRIBE HISTORY bronze_orders").show(truncate=False)

+-------+-------------------+--------------+--------------------+---------------------------------+-------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----+------------------+------------------------------------+------------------------+-----------+-----------------+-------------+-----------------------------------------------------------------------------------------------------------------------------------------+------------+------------------------------------------------+
|version|timestamp          |userId        |userName            |operation                        |operationParameters                                                                                                                                    

## Silver Layer

Clean and standardize the Bronze data by handling null values, removing duplicates, converting data types, and preparing trusted datasets for analytics.

In [0]:
silver_orders = spark.table("bronze_orders")

# Remove duplicate rows
silver_orders = silver_orders.dropDuplicates()

# Remove rows where all columns are null
silver_orders = silver_orders.na.drop(how="all")

# Fill numeric null values with 0
silver_orders = silver_orders.na.fill(0)

In [0]:
silver_orders.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("silver_orders")

In [0]:
display(spark.table("silver_orders"))

print("Silver Orders Count:", spark.table("silver_orders").count())

order_id,product_id,store_id,quantity,price,order_date
ORD-0001,PROD-018,STR-04,15,364.81,2024-01-16
ORD-0011,PROD-011,STR-06,15,42.18,2024-03-10
ORD-0014,PROD-003,STR-07,7,338.65,2024-02-12
ORD-0056,PROD-027,STR-10,5,125.04,2024-02-11
ORD-0102,PROD-010,STR-05,12,90.73,2024-02-28
ORD-0110,PROD-015,STR-05,12,238.6,2024-02-21
ORD-0113,PROD-029,STR-07,12,354.04,2024-02-11
ORD-0118,PROD-021,STR-08,15,90.83,2024-02-05
ORD-0131,PROD-025,STR-01,4,445.02,2024-02-04
ORD-0136,PROD-001,STR-09,7,336.25,2024-01-12


Silver Orders Count: 600


In [0]:
orders.printSchema()

root
 |-- order_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- store_id: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- price: double (nullable = true)
 |-- order_date: date (nullable = true)



## Gold Layer

Generate business-ready KPIs from the trusted Silver dataset.

In [0]:
from pyspark.sql.functions import sum, col

gold_daily_sales = silver_orders.groupBy("order_date").agg(
    sum(col("quantity") * col("price")).alias("total_sales"),
    sum("quantity").alias("total_quantity")
)

In [0]:
gold_daily_sales.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("gold_daily_sales")

In [0]:
display(gold_daily_sales)

order_date,total_sales,total_quantity
2024-01-16,31981.829999999998,108
2024-02-29,9577.89,37
2024-03-17,17677.88,61
2024-03-05,30377.799999999996,95
2024-01-21,17916.120000000003,60
2024-02-03,9893.599999999999,34
2024-03-22,25329.22,76
2024-03-21,15250.960000000001,49
2024-01-10,9239.560000000001,29
2024-02-04,5274.45,25


## Business Analytics using Spark SQL

In [0]:
silver_orders.createOrReplaceTempView("orders")

In [0]:
spark.sql("""
SELECT
    order_date,
    SUM(quantity * price) AS total_sales,
    SUM(quantity) AS total_quantity
FROM orders
GROUP BY order_date
ORDER BY order_date
""").show()

+----------+------------------+--------------+
|order_date|       total_sales|total_quantity|
+----------+------------------+--------------+
|2024-01-01|           17541.3|            46|
|2024-01-02|3413.2000000000003|             8|
|2024-01-03|24919.850000000002|            86|
|2024-01-04|          22339.36|            63|
|2024-01-05|16976.739999999998|            47|
|2024-01-06|15922.670000000002|            42|
|2024-01-07|          12223.42|            42|
|2024-01-08|13855.279999999999|            46|
|2024-01-09| 4824.569999999999|            23|
|2024-01-10| 9239.560000000001|            29|
|2024-01-11|          14339.36|            47|
|2024-01-12|          15021.18|            66|
|2024-01-13|          13068.48|            40|
|2024-01-14| 7987.880000000001|            22|
|2024-01-15|           26636.4|            73|
|2024-01-16|31981.829999999998|           108|
|2024-01-17|          18112.91|            72|
|2024-01-18|18698.850000000002|            76|
|2024-01-19| 

In [0]:
spark.sql("""
SELECT
    product_id,
    SUM(quantity) AS total_quantity_sold
FROM orders
GROUP BY product_id
ORDER BY total_quantity_sold DESC
LIMIT 10
""").show()

+----------+-------------------+
|product_id|total_quantity_sold|
+----------+-------------------+
|  PROD-003|                252|
|  PROD-021|                251|
|  PROD-025|                242|
|  PROD-014|                210|
|  PROD-011|                203|
|  PROD-012|                202|
|  PROD-024|                199|
|  PROD-015|                198|
|  PROD-028|                191|
|  PROD-013|                190|
+----------+-------------------+



In [0]:
spark.sql("""
SELECT
    store_id,
    SUM(quantity * price) AS revenue
FROM orders
GROUP BY store_id
ORDER BY revenue DESC
""").show()

+--------+------------------+
|store_id|           revenue|
+--------+------------------+
|  STR-06|171573.03000000003|
|  STR-08|         166960.61|
|  STR-02|152205.74000000008|
|  STR-10|150091.82999999996|
|  STR-09|145759.99000000005|
|  STR-05|144523.11000000004|
|  STR-03|         142914.16|
|  STR-07|         137636.37|
|  STR-04|133249.58000000002|
|  STR-01|114872.06000000001|
+--------+------------------+



## Pipeline Completed

The RetailPulse pipeline successfully:
- Loaded retail datasets
- Stored raw data in the Bronze layer
- Created a trusted Silver layer
- Generated Gold layer KPIs
- Performed business analytics using Spark SQL